In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from pathlib import Path

In [19]:
nlp = spacy.load("fr_core_news_lg", disable=["ner", "parser"])
nlp.max_length = 2_000_000

In [25]:
df=pd.read_csv("paquets_phrases.csv")

In [26]:
df.head()

,nom_fichier,id_paquet,phrases_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...
1,1893_20_Le_docteur_Pascal._clean.txt,1,La défense d’y entrer était formelle. C’était ...
2,1893_20_Le_docteur_Pascal._clean.txt,2,"Directeur de l’Époque, répéta-t-elle, c’est un..."
3,1893_20_Le_docteur_Pascal._clean.txt,3,"Tiens! grand-mère, les dossiers sont là-haut. ..."
4,1893_20_Le_docteur_Pascal._clean.txt,4,"Jamais elle n’entrait dans cette chambre, où i..."


In [27]:
stop_word = {
    
    "je", "tu", "il", "elle", "ils", "elles", "nous", "vous",
    "me", "te", "se", "moi", "toi", "eux", "leur", "leurs", "ses",
    "mon", "ton", "son", "notre", "votre", "cette", "ça",
    "si", "même", "où", "dont", "sans", "quand", "bien", "là", "sous",
    "être", "avoir", "faire", "dire", "aller", "voir", "pouvoir", "demander", "prendre", "mettre", "parler", "devoir", "trouver", 
    "donner", "comprendre",
    "savoir", "vouloir", "venir",
    "est", "ai", "as", "avez", "ont", "avaient", "étaient",
    "deux", "cent", "cents", "cinq", "dix", "vingt", "mille",
    "mme", "madame", " monsieur", "m", "mr", "mlle", 
    "nana", "gervaise", "louis", "lazare", "etienne", "claude", "suzanne", "mouret", "rougon", "macquart", "saccard", "lacroix", "rennée",
    "renée", "xiii", 
}

In [28]:
def nettoyer_texte(texte): #  Fonction pour nettoyer le texte : suppression des stop words, ponctuation, chiffres, espaces, et lemmatisation
    doc = nlp(texte) # Traiter le texte avec spaCy pour obtenir les tokens et leurs propriétés
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules pour une meilleure normalisation
        
        if (
            not token.is_stop # Ignorer les stop words (mots vides par défaut dans spaCy)
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces
            and token.pos_ in {"NOUN", "ADJ"} 
            and len(token.lemma_) > 2 # Ne garder que les tokens dont la longueur du lemme est supérieure à 2 caractères
            and lemme not in stop_word # Ignorer les mots présents dans la liste personnalisée de stop words
        ):
            tokens.append(lemme) # Ajouter le lemme du token en minuscules à la liste des tokens nettoyés
    
    return " ".join(tokens) # Retourner le texte nettoyé en joignant les tokens avec des espaces

df["phrases_lemm"] = df["phrases_paquet"].apply(nettoyer_texte) # Application de la fonction de nettoyage à chaque segment de texte

df[["phrases_paquet", "phrases_lemm"]].head()

,phrases_paquet,phrases_lemm
0,Dans la chaleur de l’ardente après-midi de jui...,chaleur ardent après-midi juillet salle volet ...
1,La défense d’y entrer était formelle. C’était ...,défense formel préparation spécial suite bruit...
2,"Directeur de l’Époque, répéta-t-elle, c’est un...",directeur époque vrai situation ministre père ...
3,"Tiens! grand-mère, les dossiers sont là-haut. ...",grand-mère dossier haut parole porte chambre o...
4,"Jamais elle n’entrait dans cette chambre, où i...",chambre travail close tabernacle anxiété prise...


In [29]:
df.to_csv("corpus_zola_paquets_lemm.csv", index=False)